# Monte-Carlo phase-separation reproducibility notebook

Companion notebook for<br>
**Investigating Light-Induced Phase Separation in MAPbBr<sub>1.8</sub>I<sub>1.2</sub> Using Time-Resolved X-ray Diffraction and Numerical Simulations**<br>
*Sebastian Schwartzkopff, Ivan Zaluzhnyy, Ekaterina Kneschaurek, Dmitry Lapkin, Hans Mauser, Niels Scheffczyk, Paul Zimmermann, Alexander Hinderhofer, Frederik Unger, Fabian Westermeier, Yana Vaynzof, Fabian Paulus and Frank Schreiber*<br>
<br>
JOURNAL

This notebook reproduces the three-stage simulation protocol from the paper
using **identical physics parameters** for any value of the `mul` multiplier
defined in the first code cell.  `mul` linearly scales the grid width, MC step
counts, polaron numbers, and snapshot cadence; all energetics, polaron
sigmas, and gradients are unchanged.

| `mul` | Grid       | Total MC steps     | Approx. run time |
|------:|:-----------|:-------------------|:-----------------|
| 0.2   | 706 × 100  | 2.5 × 10⁹          | ~ 1-2 h          |
| 1.0   | 706 × 500  | 1.25 × 10¹⁰        | ~ 13 h           |

`mul = 1.0` reproduces the exact published simulation.  `mul = 0.2` reproduces
the same physics at reduced cost and is suitable for verifying the install
and exploring the parameter space.

The three stages mirror the paper:
1. **Initial equilibration**: low polaron density (`N_off`), no gradients.
2. **Light-on**: polaron count raised to `N_on`, drift gradients applied.
3. **Light-off**: polaron density returns to `N_off`, gradients switched off.

After the run, lattice snapshots are converted to a diffraction-like
$q(t)$ intensity map via `concentration_to_qt` + a Voigt convolution.


## Imports

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

sys.path.insert(0, os.getcwd())

from mc import (MCPhaseSeparation, SimulationParameters,
                concentration_to_qt, voigt_blur_1d)
from mc.viz import (plot_qt, plot_lattice_snapshot, plot_lattice_grid,
                    create_animation)
from mc.progress import cycle_with_progress

N_UPDATES = 1000   # number of progress-bar refreshes per stage

%matplotlib widget


## Simulation parameters

All physics parameters (`J`, `h`, `T`, `e_barrier`, `concentration_sigma`,
`electron_polaron_sigma`, `hole_polaron_sigma`, gradients) are taken directly
from the paper and are **independent of `mul`**.  Only the grid width, step
counts, polaron numbers, and snapshot cadence scale linearly with `mul`,
holding the polaron surface density and the total number of saved frames
constant across resolutions.

Sign conventions (see `mc.engine` docstring for details):
* `J > 0` favours phase separation.
* `h = (c_e, c_h)`: polaron coupling magnitudes; the effective sign of each
  contribution is set by `electrons_at_bromide` / `holes_at_bromide`.


In [ ]:
# ── Run-size multiplier ───────────────────────────────────────────────────────
# `mul = 0.2` → grid 706 × 100, 2.5 × 10⁹ total MC steps    (smoke test, ~1-2 h)
# `mul = 1.0` → grid 706 × 500, 1.25 × 10¹⁰ total MC steps  (paper-exact, ~13 h)
# Width, step counts, polaron numbers, polaron-update cadence, and snapshot
# cadence all scale linearly with `mul`, which holds the polaron surface
# density, total snapshot count, and number of polaron updates constant.
# All physics parameters are independent of `mul`.

mul = 0.1

params = SimulationParameters(
    # ── Lattice geometry (only the width scales with `mul`) ───────────────────
    grid_shape          = (706, int(500 * mul)),
    iodide_fraction     = 0.4,                  # fraction of sites initialised as iodide
    # ── Energetics (independent of mul) ───────────────────────────────────────
    J                   = 8.0,                  # mixing-energy coupling (J>0 → phase-separating)
    h                   = (75.0, 20.0),         # (c_e, c_h) polaron-field coupling magnitudes
    T                   = 3.0,                  # effective temperature
    e_barrier           = 18.0,                 # Metropolis activation barrier
    # ── Polaron length scales (independent of mul) ────────────────────────────
    concentration_sigma         = 6.0,          # σ for lattice → concentration smearing (px)
    hole_polaron_sigma          = 3.0,          # σ of hole polaron field (px)
    electron_polaron_sigma      = 4.0,          # σ of electron polaron field (px)
    # ── Three-stage timing (linear in `mul`) ──────────────────────────────────
    equilibration_num_steps     = int(mul * int(2.5e9)),
    light_on_num_steps          = int(mul * int(5.0e9)),
    light_off_num_steps         = int(mul * int(5.0e9)),
    # ── Stage polaron counts (linear in `mul` → constant surface density) ─────
    equilibration_hole_num      = int(mul * 100),
    equilibration_electron_num  = int(mul * 100),
    light_on_hole_num           = int(mul * 5_000),
    light_on_electron_num       = int(mul * 5_000),
    light_off_hole_num          = int(mul * 100),
    light_off_electron_num      = int(mul * 100),
    # ── Stage gradients (only light-on has nonzero gradients) ─────────────────
    light_on_hole_gradient      = 5e-3,         # vertical decay rate of hole probability
    light_on_electron_gradient  = 1e-3,         # vertical decay rate of electron probability
    light_off_hole_gradient     = 0.0,
    light_off_electron_gradient = 0.0,
    # ── Sampling cadence (linear in `mul`) ────────────────────────────────────
    save_state_every            = int(mul * 10_000_000),   # keeps total snapshot count constant
    polaron_update_every        = int(mul * 25_000),       # keeps number of polaron updates constant
    # ── Polaron-field generation switches ─────────────────────────────────────
    holes_at_bromide            = False,        # holes accumulate at iodide
    electrons_at_bromide        = True,         # electrons accumulate at bromide
    # ── RNG ───────────────────────────────────────────────────────────────────
    seed                        = 10,
)

MC    = MCPhaseSeparation.from_params(params)
total = sum(params.num_steps_for_stage(s)
            for s in ("equilibration", "light_on", "light_off"))
print(f"mul = {mul}  |  grid {MC.lattice.shape}  |  {total:,} total MC steps"
      f"  |  {total // params.save_state_every + 1} snapshots")


## Three-stage protocol

The three `cycle()` blocks below correspond to *initial equilibration*,
*light on*, and *light off*.  Between cycles we call
``params.configure_for_stage(MC, stage)`` to switch protocol stage;
no re-initialisation of the lattice is needed.


In [ ]:
# Stage 1: equilibration - low polaron density, no gradients
params.configure_for_stage(MC, "equilibration")
cycle_with_progress(MC, params.equilibration_num_steps,
                    n_updates=N_UPDATES, label="equilibration")


In [ ]:
# Stage 2: light on - high polaron density, drift gradients active
params.configure_for_stage(MC, "light_on")
cycle_with_progress(MC, params.light_on_num_steps,
                    n_updates=N_UPDATES, label="light on")


In [ ]:
# Stage 3: light off - polaron density drops, gradients removed
params.configure_for_stage(MC, "light_off")
cycle_with_progress(MC, params.light_off_num_steps,
                    n_updates=N_UPDATES, label="light off")


## Animate the simulation

100 evenly-spaced frames from all accumulated snapshots, shown as a
synchronised three-panel video (lattice | hole polaron field | electron
polaron field).  The suptitle shows the MC step number for each displayed frame.


In [ ]:
create_animation(
    {
        "Lattice (0=bromide, 1=iodide)": MC.lattice_storage.astype(float),
        "Hole polarons":                 MC.hole_storage,
        "Electron polarons":             MC.electron_storage,
    },
    max_frames=100, fps=10,
    title="Simulation evolution",
    step_per_frame=MC.save_state_every,
)


## Post-process: diffraction-like *q(t)* map

`concentration_to_qt` converts each lattice snapshot (after a small Gaussian
blur to produce a smooth concentration field) into a 1-D histogram along the
reciprocal-space coordinate *q*, using the concentration → *q* lookup tables
stored in `mc/saved_arrays/` (see paper supplement for their derivation).
`voigt_blur_1d` then convolves each row with a Voigt profile whose width
parameters were fitted to the experimental diffraction peak shape.


In [ ]:
# Gaussian pre-blur (σ = 6 px) converts the binary lattice to a concentration field
blurred = gaussian_filter(MC.lattice_storage.astype(np.float64), (0, 6, 6))
qt      = concentration_to_qt(blurred, bins=2000, q_range=(0.98, 1.07), ratio=0.05)
# Voigt broadening fitted to experimental peak shape (σ = 6.88e-4, γ = 3.24e-4 Å⁻¹)
qt      = voigt_blur_1d(qt, (0.98, 1.07), 6.876864338247578e-4, 3.24e-4)
print(f"qt shape: {qt.shape}  ({qt.shape[0]} frames × {qt.shape[1]} q-bins)")


## Visualisation

In [ ]:
# q(t) heatmap of the full three-stage protocol
extent_max = qt.shape[0] * MC.save_state_every
fig, ax = plot_qt(qt, q_range=(0.98, 1.07), extent_max=extent_max,
                  vmin=5.0, vmax=None,
                  title="q(t) - full protocol")
plt.show()


In [ ]:
# Final snapshot: lattice | hole polaron field | electron polaron field
fig, axs = plot_lattice_snapshot(MC, frame=-1)
plt.show()


In [ ]:
# Time grid of the lattice evolution
fig, axs = plot_lattice_grid(MC, channel="lattice", n_cols=4)
plt.show()


## (Optional) Save to HDF5

In [ ]:
import h5py
out_path = os.path.join(os.getcwd(), "example_output.h5")
with h5py.File(out_path, "w") as f:
    ds = f.create_dataset("qt", data=qt, compression="gzip", compression_opts=4)
    ds.attrs["q_min"], ds.attrs["q_max"], ds.attrs["q_bins"] = 0.98, 1.07, 2000
    raw = f.create_group("raw")
    raw.attrs["save_state_every"] = MC.save_state_every
    raw.attrs["grid_shape"]       = list(MC.grid_shape)
    raw.create_dataset("lattice",   data=MC.lattice_storage.astype(np.uint8),
                       compression="gzip", compression_opts=4)
    raw.create_dataset("holes",     data=MC.hole_storage.astype(np.float32),
                       compression="gzip", compression_opts=4)
    raw.create_dataset("electrons", data=MC.electron_storage.astype(np.float32),
                       compression="gzip", compression_opts=4)
print("Saved to", out_path)
